# RSNA Knee — Weak Labeler dari Teks Laporan (rule-based, negation-aware, multi-bahasa)

**Kenapa rule-based, bukan LLM API:** data laporan radiologi berisi info pasien sensitif. Rules kompetisi soal pengiriman data ke API LLM eksternal belum terverifikasi mengizinkan — jadi TIDAK dipakai API hosted (Claude/GPT/dst) sesuai instruksi. Semua pemrosesan teks di sini murni regex + heuristik lokal, tidak ada panggilan API eksternal sama sekali.

**Tujuan:** ubah 4.407 studi (100% punya `Report`, tapi cuma 58/4.407 = 1.3% punya label gold) jadi weak-label untuk 4.349 studi sisanya, divalidasi dulu presisi/recall-nya terhadap 58 studi gold sebelum dipakai skala penuh.

**Referensi pola:** deteksi negasi/uncertainty per-klausa, bukan cuma keyword match polos — meniru pendekatan `pilkwang/rsna-knee-baseline-v1` (lihat `docs/RESEARCH_NOTES.md` bagian 1.4).

## 0. Setup

In [ ]:
import os, sys, re, json, time, unicodedata
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings_off = __import__("warnings"); warnings_off.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:6.1f}s] {msg}", flush=True)

OUT_DIR = Path("labeler_outputs")
OUT_DIR.mkdir(exist_ok=True)
ACCENT = "#3B7DD8"; ACCENT2 = "#D8763B"
PALETTE = ["#3B7DD8", "#D8763B", "#4CA36B", "#B24C63", "#7A5CC0", "#C9A227", "#4CA3A3", "#8C8C8C"]
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.grid": True, "grid.color": "#e6e6e6"})
def savefig(fig, name, **kw):
    fig.tight_layout(); fig.savefig(OUT_DIR / name, dpi=140, bbox_inches="tight", **kw)


In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                    Path("../data"), Path("data")]
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/*/train.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and (p/"train.csv").exists()), None) or CANDIDATE_ROOTS[0]
tlog(f"DATA_DIR: {DATA_DIR}")

train = pd.read_csv(DATA_DIR / "train.csv").drop_duplicates(subset="StudyInstanceUID").reset_index(drop=True)
LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if c in train.columns]
gold_mask = train[LABEL_COLS].notna().any(axis=1)
GOLD = train.loc[gold_mask].reset_index(drop=True)
tlog(f"train: {train.shape} | studi gold-label: {len(GOLD)}")


## 1. Lexicon: cue negasi/uncertain/normal (per-bahasa) + istilah target (per-label, per-bahasa)

**v1 -- akan divalidasi empiris terhadap 58 studi gold di bawah, lalu diperbaiki berdasarkan hasil (bagian mana yang recall-nya rendah).** 9 bahasa dicakup sesuai temuan EDA: en, es, tr, hr, el, de, bg, nl, fr.

In [ ]:
# Cue negasi/uncertain/normal bersifat GLOBAL per-bahasa (bukan per-label) -- kelas kata tertutup,
# jadi tidak perlu didaftar ulang utk tiap 12 label (meniru pola pilkwang: _polarity() generik).
NEGATION = {
    "en": ["no ", "not ", "without", "absence of", "negative for", "no evidence of", "rule(d)? out", "free of", "excludes?"],
    "es": ["no ", "sin ", "ausencia de", "sin evidencia de", "descarta(do)?", "libre de"],
    "tr": ["yok", "izlenme(di|mekte|z)", "saptanma(dı|makta)", "g\u00f6r\u00fclme(di|mekte)"],
    "hr": ["nema", "bez ", "ne ", "isklju\u010den"],
    "el": ["\u03cc\u03c7\u03b9", "\u03c7\u03c9\u03c1\u03af\u03c2", "\u03b4\u03b5\u03bd "],
    "de": ["kein", "keine", "ohne ", "nicht ", "negativ f\u00fcr", "frei von"],
    "bg": ["\u043d\u0435 ", "\u0431\u0435\u0437 ", "\u043b\u0438\u043f\u0441\u0432\u0430"],
    "nl": ["geen ", "niet ", "zonder ", "afwezig"],
    "fr": ["pas de ", "sans ", "absence de", "aucun", "n['\u2019]a pas"],
}
UNCERTAIN = {
    "en": ["possible", "probable", "cannot exclude", "suspicious for", "equivocal", "may represent", "likely", "questionable"],
    "es": ["posible", "probable", "no se puede excluir", "sospechoso de", "dudoso"],
    "tr": ["olas\u0131", "\u015f\u00fcpheli", "d\u0131\u015flanamaz"],
    "hr": ["mogu\u0107", "sumnjiv", "ne mo\u017ee se isklju\u010diti"],
    "el": ["\u03c0\u03b9\u03b8\u03b1\u03bd\u03cc\u03c2", "\u03cd\u03c0\u03bf\u03c0\u03c4\u03bf\u03c2"],
    "de": ["m\u00f6glich", "wahrscheinlich", "verd\u00e4chtig auf", "nicht auszuschlie\u00dfen", "fraglich"],
    "bg": ["\u0432\u044a\u0437\u043c\u043e\u0436\u0435\u043d", "\u0441\u044a\u043c\u043d\u0438\u0442\u0435\u043b\u0435\u043d"],
    "nl": ["mogelijk", "waarschijnlijk", "verdacht voor"],
    "fr": ["possible", "probable", "suspect de", "ne peut exclure", "\u00e9voquant"],
}
NORMAL = {
    "en": ["normal", "intact", "unremarkable", "within normal limits", "no abnormality"],
    "es": ["normal", "intacto", "sin alteraciones", "dentro de l\u00edmites normales"],
    "tr": ["normal", "sa\u011flam", "do\u011fal"],
    "hr": ["normalan", "uredan", "o\u010duvan"],
    "el": ["\u03c6\u03c5\u03c3\u03b9\u03bf\u03bb\u03bf\u03b3\u03b9\u03ba\u03cc\u03c2", "\u03b1\u03ba\u03ad\u03c1\u03b1\u03b9\u03bf\u03c2"],
    "de": ["normal", "intakt", "unauff\u00e4llig", "regelrecht"],
    "bg": ["\u043d\u043e\u0440\u043c\u0430\u043b\u0435\u043d", "\u0437\u0430\u043f\u0430\u0437\u0435\u043d"],
    "nl": ["normaal", "intact", "ongestoord"],
    "fr": ["normal", "intact", "sans particularit\u00e9"],
}

# Istilah POSITIF per label per bahasa -- cognate medis lintas-bahasa dimanfaatkan sebisa mungkin.
TARGET_TERMS = {
    "ACL": {"en": ["acl", "anterior cruciate"], "es": ["lca", "ligamento cruzado anterior"],
            "tr": ["\u00f6n \u00e7apraz ba\u011f", "acl"], "hr": ["prednji ukri\u017eeni ligament", "acl"],
            "el": ["\u03c0\u03c1\u03cc\u03c3\u03b8\u03b9\u03bf\u03c2 \u03c7\u03b9\u03b1\u03c3\u03c4\u03cc\u03c2", "acl"],
            "de": ["vkb", "vorderes kreuzband"], "bg": ["\u043f\u0440\u0435\u0434\u043d\u0430 \u043a\u0440\u044a\u0441\u0442\u043d\u0430 \u0432\u0440\u044a\u0437\u043a\u0430", "acl"],
            "nl": ["vkb", "voorste kruisband"], "fr": ["lca", "ligament crois\u00e9 ant\u00e9rieur"]},
    "MCL": {"en": ["mcl", "medial collateral"], "es": ["lcm", "ligamento colateral medial"],
            "tr": ["i\u00e7 yan ba\u011f", "mcl"], "hr": ["medijalni kolateralni ligament"],
            "el": ["\u03ad\u03c3\u03c9 \u03c0\u03bb\u03ac\u03b3\u03b9\u03bf\u03c2 \u03c3\u03cd\u03bd\u03b4\u03b5\u03c3\u03bc\u03bf\u03c2"],
            "de": ["mcl", "mediales seitenband"], "bg": ["\u043c\u0435\u0434\u0438\u0430\u043b\u0435\u043d \u043a\u043e\u043b\u0430\u0442\u0435\u0440\u0430\u043b\u0435\u043d \u043b\u0438\u0433\u0430\u043c\u0435\u043d\u0442"],
            "nl": ["mcl", "mediale collaterale band"], "fr": ["lcm", "ligament collat\u00e9ral m\u00e9dial"]},
    "Medial Meniscus": {"en": ["medial meniscus"], "es": ["menisco medial", "menisco interno"],
            "tr": ["i\u00e7 menisk\u00fcs", "medial menisk\u00fcs"], "hr": ["medijalni meniskus"],
            "el": ["\u03ad\u03c3\u03c9 \u03bc\u03b7\u03bd\u03af\u03c3\u03ba\u03bf\u03c2"],
            "de": ["medialer meniskus", "innenmeniskus"], "bg": ["\u043c\u0435\u0434\u0438\u0430\u043b\u0435\u043d \u043c\u0435\u043d\u0438\u0441\u043a\u0443\u0441"],
            "nl": ["mediale meniscus"], "fr": ["m\u00e9nisque m\u00e9dial", "m\u00e9nisque interne"]},
    "Lateral Meniscus": {"en": ["lateral meniscus"], "es": ["menisco lateral", "menisco externo"],
            "tr": ["d\u0131\u015f menisk\u00fcs", "lateral menisk\u00fcs"], "hr": ["lateralni meniskus"],
            "el": ["\u03ad\u03be\u03c9 \u03bc\u03b7\u03bd\u03af\u03c3\u03ba\u03bf\u03c2"],
            "de": ["lateraler meniskus", "au\u00dfenmeniskus"], "bg": ["\u043b\u0430\u0442\u0435\u0440\u0430\u043b\u0435\u043d \u043c\u0435\u043d\u0438\u0441\u043a\u0443\u0441"],
            "nl": ["laterale meniscus"], "fr": ["m\u00e9nisque lat\u00e9ral", "m\u00e9nisque externe"]},
    # OA di 3 label ini SENGAJA tidak diisi -- ditangani terpisah lewat OA_INDICATOR x LOCATION
    # co-occurrence di bawah (lihat sel extractor), karena laporan asli menyebut cartilage loss/
    # osteophyte/joint space narrowing dan kata lokasi (medial/lateral/patellofemoral) sbg klausa
    # terpisah, bukan satu frasa gabungan tetap -- terverifikasi langsung dari sampel laporan asli.
    "Medial OA": {}, "Lateral OA": {}, "PF OA": {},
    "Effusion": {"en": ["effusion", "joint effusion"], "es": ["derrame", "derrame articular"],
            "tr": ["ef\u00fczyon", "s\u0131v\u0131 birikimi"], "hr": ["izljev"],
            "el": ["\u03c3\u03c5\u03bb\u03bb\u03bf\u03b3\u03ae \u03c5\u03b3\u03c1\u03bf\u03cd"],
            "de": ["erguss", "gelenkerguss"], "bg": ["\u0438\u0437\u043b\u0438\u0432", "\u0441\u0442\u0430\u0432\u0435\u043d \u0438\u0437\u043b\u0438\u0432"],
            "nl": ["effusie", "vochtophoping", "gewrichtsvocht"], "fr": ["\u00e9panchement"]},
    "Synovitis": {"en": ["synovitis", "synovial thickening"], "es": ["sinovitis"], "tr": ["sinovit"],
            "hr": ["sinovitis"], "el": ["\u03c3\u03c5\u03bd\u03bf\u03b2\u03af\u03c4\u03b9\u03b4\u03b1"],
            "de": ["synovitis", "synoviale verdickung"], "bg": ["\u0441\u0438\u043d\u043e\u0432\u0438\u0442"],
            "nl": ["synovitis"], "fr": ["synovite"]},
    "Baker's": {"en": ["baker's cyst", "bakers cyst", "popliteal cyst"], "es": ["quiste de baker", "quiste popl\u00edteo"],
            "tr": ["baker kisti", "popliteal kist"], "hr": ["bakerova cista"],
            "el": ["\u03ba\u03cd\u03c3\u03c4\u03b7 baker"], "de": ["baker-zyste", "poplitealzyste"],
            "bg": ["\u0431\u0435\u0439\u043a\u044a\u0440\u043e\u0432\u0430 \u043a\u0438\u0441\u0442\u0430"],
            "nl": ["bakercyste"], "fr": ["kyste de baker", "kyste poplit\u00e9"]},
    "Contusion": {"en": ["contusion", "bone bruise", "bone marrow edema", "marrow edema"],
            "es": ["contusi\u00f3n", "edema \u00f3seo", "edema medular"],
            "tr": ["kont\u00fczyon", "kemik ili\u011fi \u00f6demi"], "hr": ["kontuzija", "ko\u0161tani edem"],
            "el": ["\u03b8\u03bb\u03ac\u03c3\u03b7"], "de": ["kontusion", "knochenmark\u00f6dem", "bone bruise"],
            "bg": ["\u043a\u043e\u043d\u0442\u0443\u0437\u0438\u044f", "\u043a\u043e\u0441\u0442\u043d\u043e\u043c\u043e\u0437\u044a\u0447\u0435\u043d \u0435\u0434\u0435\u043c"],
            "nl": ["contusie", "beenmergoedeem"], "fr": ["contusion", "\u0153d\u00e8me osseux"]},
    "Fracture": {"en": ["fracture"], "es": ["fractura"], "tr": ["k\u0131r\u0131k", "frakt\u00fcr"],
            "hr": ["fraktura", "prijelom"], "el": ["\u03ba\u03ac\u03c4\u03b1\u03b3\u03bc\u03b1"],
            "de": ["fraktur", "bruch"], "bg": ["\u0444\u0440\u0430\u043a\u0442\u0443\u0440\u0430"],
            "nl": ["fractuur"], "fr": ["fracture"]},
}
# OA (Medial/Lateral/PF) ditangani via co-occurrence: (istilah indikator OA) + (kata lokasi) dalam klausa sama.
# Diverifikasi dari sampel laporan asli: cartilage loss/osteophyte/joint space narrowing muncul terpisah
# dari kata lokasi, bukan satu frasa gabungan (mis. Yunani "\u03b5\u03be\u03ac\u03bb\u03b5\u03b9\u03c8\u03b7 \u03c4\u03bf\u03c5 \u03b1\u03c1\u03b8\u03c1\u03b9\u03ba\u03bf\u03cd \u03c7\u03cc\u03bd\u03b4\u03c1\u03bf\u03c5" dekat "\u03ad\u03c3\u03c9 \u03b4\u03b9\u03b1\u03bc\u03ad\u03c1\u03b9\u03c3\u03bc\u03b1").
OA_INDICATOR = {
    "en": ["osteoarthritis", "\bcartilage loss\b", "chondral loss", "chondral thinning", "chondromalacia",
           "joint space narrowing", "osteophyte", "degenerative change", "cartilage thinning", "full.thickness cartilage"],
    "es": ["artrosis", "osteoartritis", "p\u00e9rdida de cart\u00edlago", "condromalacia", "pinzamiento articular",
           "osteofito", "cambios degenerativos", "adelgazamiento del cart\u00edlago"],
    "tr": ["osteoartrit", "artroz", "k\u0131k\u0131rdak kayb\u0131", "kondromalazi", "eklem aral\u0131\u011f\u0131 daralmas\u0131",
           "osteofit", "dejeneratif"],
    "hr": ["osteoartritis", "artroza", "gubitak hrskavice", "hondromalacija", "su\u017eenje zglobnog prostora",
           "osteofit", "degenerativn"],
    "el": ["\u03bf\u03c3\u03c4\u03b5\u03bf\u03b1\u03c1\u03b8\u03c1\u03af\u03c4\u03b9\u03b4\u03b1", "\u03b1\u03c0\u03ce\u03bb\u03b5\u03b9\u03b1 \u03c7\u03cc\u03bd\u03b4\u03c1\u03bf\u03c5",
           "\u03b5\u03be\u03ac\u03bb\u03b5\u03b9\u03c8\u03b7.{0,20}\u03c7\u03cc\u03bd\u03b4\u03c1", "\u03c7\u03bf\u03bd\u03b4\u03c1\u03bf\u03bc\u03b1\u03bb\u03b1\u03ba\u03af\u03b1",
           "\u03bf\u03c3\u03c4\u03b5\u03cc\u03c6\u03c5\u03c4\u03b1", "\u03b5\u03ba\u03c6\u03c5\u03bb\u03b9\u03c3\u03c4\u03b9\u03ba"],
    "de": ["arthrose", "knorpelverlust", "chondromalazie", "gelenkspalt.{0,15}verschm\u00e4lert", "osteophyt",
           "degenerativ", "knorpelschaden", "knorpelaufbrauch"],
    "bg": ["\u043e\u0441\u0442\u0435\u043e\u0430\u0440\u0442\u0440\u0438\u0442", "\u0430\u0440\u0442\u0440\u043e\u0437\u0430", "\u0437\u0430\u0433\u0443\u0431\u0430 \u043d\u0430 \u0445\u0440\u0443\u0449\u044f\u043b",
           "\u0445\u043e\u043d\u0434\u0440\u043e\u043c\u0430\u043b\u0430\u0446\u0438\u044f", "\u043e\u0441\u0442\u0435\u043e\u0444\u0438\u0442", "\u0434\u0435\u0433\u0435\u043d\u0435\u0440\u0430\u0442\u0438\u0432\u043d"],
    "nl": ["artrose", "kraakbeenverlies", "kraakbeenlijden", "chondromalacie", "gewrichtsspleet.{0,15}versmalling",
           "osteofyt", "kraakbeenverval", "degeneratie"],
    "fr": ["arthrose", "perte de cartilage", "chondromalacie", "pincement articulaire", "ost\u00e9ophyte",
           "d\u00e9g\u00e9n\u00e9ratif", "amincissement du cartilage"],
}
LOCATION = {
    "Medial OA": {
        "en": ["medial compartment", "medial femorotibial", "medial tibiofemoral", "medial tibial plateau",
               "medial femoral condyle", "\bmedial\b"],
        "es": ["compartimento medial", "femorotibial medial", "\bmedial\b"],
        "tr": ["medial kompartman", "i\u00e7 kompartman", "\bmedial\b", "i\u00e7 "],
        "hr": ["medijalni odjeljak", "medijaln"],
        "el": ["\u03ad\u03c3\u03c9 \u03b4\u03b9\u03b1\u03bc\u03ad\u03c1\u03b9\u03c3\u03bc\u03b1", "\u03ad\u03c3\u03c9 \u03ba\u03bd\u03b7\u03bc\u03b9\u03b1\u03af", "\u03ad\u03c3\u03c9 \u03bc\u03b7\u03c1\u03b9\u03b1\u03af"],
        "de": ["medial", "innenseitig"],
        "bg": ["\u043c\u0435\u0434\u0438\u0430\u043b\u0435\u043d \u043e\u0442\u0434\u0435\u043b", "\u043c\u0435\u0434\u0438\u0430\u043b"],
        "nl": ["mediale compartiment", "mediale femorotibiale", "mediaal"],
        "fr": ["compartiment m\u00e9dial", "f\u00e9moro-tibial m\u00e9dial", "m\u00e9dial"],
    },
    "Lateral OA": {
        "en": ["lateral compartment", "lateral femorotibial", "lateral tibiofemoral", "lateral tibial plateau",
               "lateral femoral condyle", "\blateral\b"],
        "es": ["compartimento lateral", "femorotibial lateral", "\blateral\b"],
        "tr": ["lateral kompartman", "d\u0131\u015f kompartman", "\blateral\b", "d\u0131\u015f "],
        "hr": ["lateralni odjeljak", "lateraln"],
        "el": ["\u03ad\u03be\u03c9 \u03b4\u03b9\u03b1\u03bc\u03ad\u03c1\u03b9\u03c3\u03bc\u03b1", "\u03ad\u03be\u03c9 \u03ba\u03bd\u03b7\u03bc\u03b9\u03b1\u03af", "\u03ad\u03be\u03c9 \u03bc\u03b7\u03c1\u03b9\u03b1\u03af"],
        "de": ["lateral", "au\u00dfenseitig"],
        "bg": ["\u043b\u0430\u0442\u0435\u0440\u0430\u043b\u0435\u043d \u043e\u0442\u0434\u0435\u043b", "\u043b\u0430\u0442\u0435\u0440\u0430\u043b\u0435\u043d"],
        "nl": ["laterale compartiment", "laterale femorotibiale", "lateraal"],
        "fr": ["compartiment lat\u00e9ral", "f\u00e9moro-tibial lat\u00e9ral", "lat\u00e9ral"],
    },
    "PF OA": {
        "en": ["patellofemoral", "retropatellar", "trochle", "patella"],
        "es": ["patelofemoral", "femoropatelar", "retropatelar", "tr\u00f3clea", "patela"],
        "tr": ["patellofemoral", "troklea", "patella"],
        "hr": ["patelofemoraln", "trohlea", "patela"],
        "el": ["\u03bc\u03b7\u03c1\u03bf\u03bc\u03b7\u03c1\u03b9\u03b1\u03af", "\u03c4\u03c1\u03bf\u03c7\u03bb\u03af\u03b1", "\u03b5\u03c0\u03b9\u03b3\u03bf\u03bd\u03b1\u03c4\u03af\u03b4\u03b1"],
        "de": ["retropatellar", "femoropatellar", "trochlea", "patella"],
        "bg": ["\u043f\u0430\u0442\u0435\u043b\u043e\u0444\u0435\u043c\u043e\u0440\u0430\u043b\u0435\u043d", "\u0442\u0440\u043e\u0445\u043b\u0435\u044f", "\u043f\u0430\u0442\u0435\u043b\u0430"],
        "nl": ["patellofemorale", "retropatellair", "trochlea", "patella"],
        "fr": ["f\u00e9moro-patellaire", "r\u00e9tro-patellaire", "trochl\u00e9e", "patella"],
    },
}
OA_LABELS = set(LOCATION.keys())

LANGS = list(NEGATION.keys())
tlog(f"Lexicon dimuat: {len(TARGET_TERMS)} label x {len(LANGS)} bahasa (+ {len(OA_LABELS)} label OA via co-occurrence)")


## 2. Fungsi ekstraksi: segmentasi klausa + deteksi polaritas per-kemunculan istilah

In [ ]:
_TURKISH_FOLD = str.maketrans({"\u0131": "i", "\u0130": "i", "I": "i"})  # dotless/dotted i Turki -- fold sebelum lower()

def _norm(text, lang):
    t = text.translate(_TURKISH_FOLD) if lang == "tr" else text
    t = t.lower()
    t = unicodedata.normalize("NFKC", t)
    return t

def _compile_any(cues):
    return re.compile("|".join(cues), re.IGNORECASE)

_NEG_RE = {l: _compile_any(v) for l, v in NEGATION.items()}
_UNC_RE = {l: _compile_any(v) for l, v in UNCERTAIN.items()}
_NRM_RE = {l: _compile_any(v) for l, v in NORMAL.items()}

CLAUSE_SPLIT = re.compile(r"[.;\n]|(?:\bpero\b|\bbut\b|\bama\b|\baber\b|\bmais\b)", re.IGNORECASE)

def split_clauses(text):
    return [c.strip() for c in CLAUSE_SPLIT.split(text) if c and c.strip()]

def extract_labels(report, lang):
    """Return dict {label: 0/1/None} untuk satu laporan. None = tidak disebut sama sekali (silence)."""
    if not isinstance(report, str) or not report.strip():
        return {t: None for t in TARGET_TERMS}
    lang = lang if lang in TARGET_TERMS["ACL"] else "en"  # fallback bahasa tak dikenal -> coba pola Inggris
    text = _norm(report, lang)
    clauses = split_clauses(text)
    neg_re, unc_re, nrm_re = _NEG_RE.get(lang, _NEG_RE["en"]), _UNC_RE.get(lang, _UNC_RE["en"]), _NRM_RE.get(lang, _NRM_RE["en"])

    out = {}
    oa_ind_re = _compile_any(OA_INDICATOR.get(lang, OA_INDICATOR["en"]))

    for target in list(TARGET_TERMS.keys()):
        if target in OA_LABELS:
            # co-occurrence: klausa harus punya (indikator OA) DAN (kata lokasi spesifik label ini)
            loc_terms = LOCATION[target].get(lang, LOCATION[target]["en"])
            term_re = _compile_any(loc_terms)
            requires_indicator = True
        else:
            terms = TARGET_TERMS[target].get(lang, TARGET_TERMS[target]["en"])
            term_re = _compile_any([re.escape(t) for t in terms])
            requires_indicator = False

        verdicts = []
        for clause in clauses:
            if not term_re.search(clause):
                continue
            if requires_indicator and not oa_ind_re.search(clause):
                continue   # kata lokasi ada tapi tidak ada indikator OA di klausa yg sama -> lewati
            if unc_re.search(clause):
                verdicts.append(None)          # uncertain -> jangan dihitung positif/negatif
            elif neg_re.search(clause):
                verdicts.append(0)
            elif nrm_re.search(clause):
                verdicts.append(0)
            else:
                verdicts.append(1)
        if not verdicts:
            out[target] = None                  # silence: term tidak pernah disebut
        elif 1 in verdicts:
            out[target] = 1                      # positif menang kalau ada klausa positif eksplisit manapun
        elif all(v == 0 for v in verdicts if v is not None):
            out[target] = 0
        else:
            out[target] = None
    return out


## 3. Validasi ke 58 studi gold (precision/recall/agreement per label)

Mengukur seberapa bagus extractor SEBELUM dipakai skala penuh ke 4.349 studi sisanya -- persis pola yang dipakai kompetitor top.

In [ ]:
GOLD_LANG = GOLD.copy()
# bahasa per studi -- pakai langdetect kalau tersedia, fallback 'en'
try:
    import importlib
    try:
        importlib.import_module("langdetect")
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q langdetect")
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42
    def _lang(t):
        try:
            return detect(t) if isinstance(t, str) and len(t.strip()) > 8 else "en"
        except Exception:
            return "en"
    GOLD_LANG["lang"] = GOLD_LANG["Report"].apply(_lang)
except Exception as e:
    print(f"[warn] langdetect tidak tersedia ({e}), fallback semua 'en'")
    GOLD_LANG["lang"] = "en"

pred_rows = []
for _, row in GOLD_LANG.iterrows():
    pred = extract_labels(row["Report"], row["lang"])
    pred["StudyInstanceUID"] = row["StudyInstanceUID"]
    pred["lang"] = row["lang"]
    pred_rows.append(pred)
PRED = pd.DataFrame(pred_rows).set_index("StudyInstanceUID")

rows = []
for label in LABEL_COLS:
    truth = GOLD.set_index("StudyInstanceUID")[label]
    pred = PRED[label]
    both = pd.concat([truth, pred], axis=1, keys=["truth", "pred"]).dropna(subset=["truth"])
    n = len(both)
    silence = both["pred"].isna().mean()
    scored = both.dropna(subset=["pred"])
    if len(scored):
        agree = (scored["truth"] == scored["pred"]).mean()
        tp = ((scored["truth"] == 1) & (scored["pred"] == 1)).sum()
        fp = ((scored["truth"] == 0) & (scored["pred"] == 1)).sum()
        fn = ((scored["truth"] == 1) & (scored["pred"] == 0)).sum()
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
    else:
        agree = precision = recall = float("nan")
    rows.append({"label": label, "n_gold": n, "silence_rate": round(silence, 3),
                 "agreement": round(agree, 3), "precision": round(precision, 3), "recall": round(recall, 3)})

VALIDATION = pd.DataFrame(rows)
VALIDATION.to_csv(OUT_DIR / "validation_per_label.csv", index=False)
tlog("Validasi selesai.")
display(VALIDATION)
print(f"\nRata-rata: agreement={VALIDATION['agreement'].mean():.3f} | "
      f"precision={VALIDATION['precision'].mean():.3f} | recall={VALIDATION['recall'].mean():.3f} | "
      f"silence={VALIDATION['silence_rate'].mean():.3f}")


In [ ]:
# Breakdown per-bahasa -- cari bahasa mana yang lemah (silence rate tinggi = lexicon kurang lengkap di situ)
lang_rows = []
for lang, grp in GOLD_LANG.groupby("lang"):
    if len(grp) < 3:
        continue
    idx = grp["StudyInstanceUID"]
    sil = PRED.loc[idx, LABEL_COLS].isna().mean().mean()
    lang_rows.append({"lang": lang, "n_studi": len(grp), "silence_rate_avg": round(sil, 3)})
LANG_BREAKDOWN = pd.DataFrame(lang_rows).sort_values("silence_rate_avg", ascending=False)
display(LANG_BREAKDOWN)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(VALIDATION["label"], VALIDATION["agreement"], color=ACCENT, label="agreement")
ax.set_xticklabels(VALIDATION["label"], rotation=45, ha="right")
ax.set_ylabel("agreement dgn gold label")
ax.set_title("Akurasi weak-labeler per pathology (validasi 58 studi gold)")
ax.axhline(VALIDATION["agreement"].mean(), color=ACCENT2, linestyle="--", label=f"rata-rata {VALIDATION['agreement'].mean():.3f}")
ax.legend()
savefig(fig, "validation_agreement_per_label.png")
plt.show()


## 4. Terapkan ke seluruh 4.349 studi non-gold (jadi weak-label training set)

In [ ]:
NONGOLD = train.loc[~gold_mask].reset_index(drop=True)
try:
    NONGOLD["lang"] = NONGOLD["Report"].apply(_lang)
except NameError:
    NONGOLD["lang"] = "en"

all_rows = []
for i, row in NONGOLD.iterrows():
    pred = extract_labels(row["Report"], row["lang"])
    pred["StudyInstanceUID"] = row["StudyInstanceUID"]
    pred["lang"] = row["lang"]
    pred["source"] = "weak_rule_based"
    all_rows.append(pred)
    if (i + 1) % 1000 == 0:
        tlog(f"  {i+1}/{len(NONGOLD)} studi diproses")

WEAK = pd.DataFrame(all_rows)
GOLD_OUT = GOLD[["StudyInstanceUID"] + LABEL_COLS].copy()
GOLD_OUT["lang"] = GOLD_LANG["lang"].values
GOLD_OUT["source"] = "gold"

FULL_LABELS = pd.concat([GOLD_OUT, WEAK[["StudyInstanceUID"] + LABEL_COLS + ["lang", "source"]]], ignore_index=True)
FULL_LABELS.to_csv(OUT_DIR / "weak_labels_full.csv", index=False)
tlog(f"SELESAI. weak_labels_full.csv: {FULL_LABELS.shape}")

print("\nCakupan (non-null) per label di studi non-gold (dari total {} studi):".format(len(NONGOLD)))
print((WEAK[LABEL_COLS].notna().mean() * 100).round(1))
print("\nDistribusi positif (dari yang ke-cover) per label:")
print((WEAK[LABEL_COLS] == 1).sum())


## Catatan & keterbatasan

- **v1 lexicon** -- tabel validasi di bagian 3 & 4 adalah alat ukur untuk perbaikan iteratif berikutnya, bukan hasil final. Label dengan `silence_rate` tinggi atau `recall` rendah butuh perluasan istilah.
- Studi tanpa kemunculan istilah target sama sekali (`silence`) sengaja diberi `None` (bukan 0) -- lebih aman untuk weak-label daripada memaksa asumsi negatif.
- Bahasa di luar 9 yang dicakup fallback ke pola bahasa Inggris (bisa jadi tidak akurat).
- Output `weak_labels_full.csv` siap dipakai training via `kernel_sources` di notebook modeling.